In [7]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor



In [10]:
import pandas as pd

fc_df = pd.read_csv("fc_NTER2.csv")

print(fc_df.shape)
fc_df.head()
fc_df.info()

(46482, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46482 entries, 0 to 46481
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     46482 non-null  object 
 1   GeneName      46482 non-null  object 
 2   Phosphosite   46482 non-null  object 
 3   Perturbation  46482 non-null  object 
 4   CellLine      46482 non-null  object 
 5   FC            46482 non-null  float64
dtypes: float64(1), object(5)
memory usage: 2.1+ MB


In [11]:
ksea_df = pd.read_csv("ksea_NTERA2.csv")

print(ksea_df.shape)
ksea_df.head()
ksea_df.info()

(13841, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13841 entries, 0 to 13840
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     13841 non-null  object 
 1   GeneName      13841 non-null  object 
 2   Perturbation  13841 non-null  object 
 3   CellLine      13841 non-null  object 
 4   KSEA_z_score  10486 non-null  float64
dtypes: float64(1), object(4)
memory usage: 540.8+ KB


In [12]:
merged_df = fc_df.merge(
    ksea_df,
    on=["UniprotID", "GeneName", "Perturbation", "CellLine"],
    how="inner"
)

print(merged_df.shape)
merged_df.head()

(46452, 7)


,UniprotID,GeneName,Phosphosite,Perturbation,CellLine,FC,KSEA_z_score
0,P12931,SRC,SRC(S75),AZD6482,NTERA2,-0.099699,1.812098
1,P12931,SRC,SRC(S75),CAL101,NTERA2,-1.108666,1.146269
2,P12931,SRC,SRC(S75),GDC0941,NTERA2,0.077299,1.631653
3,P12931,SRC,SRC(S75),HS173,NTERA2,-1.570072,1.744843
4,P12931,SRC,SRC(S75),PIK294,NTERA2,-1.099170,-0.228499


In [13]:
# Remove rows with missing KSEA
usable_df = merged_df.dropna(subset=["KSEA_z_score"]).copy()

print(usable_df.shape)

(36082, 7)


In [15]:
#Count phosphosites per kinase

phosphosite_count = (
    usable_df
    .groupby("UniprotID")["Phosphosite"]
    .nunique()
    .reset_index()

)

phosphosite_count.columns = ["UniprotID", "Num_Phosphosites"]

phosphosite_count

,UniprotID,Num_Phosphosites
0,O00418,5
1,O00506,1
2,O14578,6
3,O14733,2
4,O14757,2
...,...,...
167,Q9Y463,1
168,Q9Y4K4,2
169,Q9Y572,2
170,Q9Y5S2,1


In [16]:
#keep kinases with at least 3 phosphosites

usable_kinases = phosphosite_count[
    phosphosite_count["Num_Phosphosites"] >= 3
]

print(usable_kinases.shape)

usable_kinases

(85, 2)


,UniprotID,Num_Phosphosites
0,O00418,5
2,O14578,6
7,O15075,7
11,O43318,4
12,O43353,4
...,...,...
156,Q9NYV4,22
162,Q9UKE5,6
164,Q9Y2K2,8
165,Q9Y2U5,8


In [17]:
usable_kinases.info()

<class 'pandas.core.frame.DataFrame'>
Index: 85 entries, 0 to 166
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   UniprotID         85 non-null     object
 1   Num_Phosphosites  85 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 2.0+ KB


In [18]:
# store one feature matrix for each kinase
feature_matrices = {}

In [19]:
#Build one matrix per kinase

for kinase_id in usable_kinases["UniprotID"]:
    
    #Get data for one kinase
    kinase_data = usable_df[
        usable_df["UniprotID"] ==kinase_id 
    ].copy()

    # Convert long format to wide format
    kinase_matrix = kinase_data.pivot_table(
        index="Perturbation",
        columns="Phosphosite",
        values="FC",
        aggfunc="first" 
    )

    #Get one KSEA value for each perturbation
    ksea = (
        kinase_data[
            ["Perturbation","KSEA_z_score"] 
        ]
        .drop_duplicates()
        .set_index("Perturbation") 
    )

    # Join FC features with KSEA target
    kinase_matrix = kinase_matrix.join(ksea)

    #Store using Uniprot ID
    feature_matrices[kinase_id] = kinase_matrix

print(f"Created {len(feature_matrices)} feature matrices.")

Created 85 feature matrices.


In [20]:
print(len(feature_matrices))

85


In [21]:
#Let's inspect one matrix

first_kinase = list(feature_matrices.keys())[0]

print(first_kinase)

feature_matrices[first_kinase]

O00418


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69),KSEA_z_score
Perturbation,,,,,,
AC220,-0.669597,-1.389779,-0.107403,-0.118885,-0.230559,-6.199636
AT13148,-0.170124,1.309982,-2.477483,-2.136299,-1.968249,3.888057
AZ20,-0.209610,-0.082364,-0.908485,-0.621383,-0.876680,3.689873
AZD1480,-0.467337,0.795807,0.451792,0.352203,0.258493,-0.395978
AZD3759,-1.439665,-1.421963,0.442492,0.943424,-2.161574,-0.342026
...,...,...,...,...,...,...
Torin,-0.358349,-0.181748,-1.762406,-4.352456,-1.671599,0.100930
Trametinib,0.464216,-0.687976,0.662005,0.562184,0.533487,0.359160
U73122,0.425191,0.607640,0.325120,0.423954,0.715446,1.819243


In [22]:
# Create dictionaries for x and y

# Store features and targets separately
X_data = {}
y_data = {}

In [23]:
# Split every kinase matrix

for kinase_id, matrix in feature_matrices.items():

    # Remove rows where target is missing
    matrix = matrix.dropna(subset=["KSEA_z_score"])

    # Features (all phosphosite FC values)
    X = matrix.drop(columns=["KSEA_z_score"])

    # Target (kinase activity)
    y = matrix["KSEA_z_score"]

    # Store
    X_data[kinase_id] = X
    y_data[kinase_id] = y

print(f"Prepared X and y for {len(X_data)} kinases.")

Prepared X and y for 85 kinases.


In [24]:
# Inspect one kinase

first_kinase = list(X_data.keys())[0]

print("Kinase:", first_kinase)

print("\nX shape:", X_data[first_kinase].shape)
print("y shape:", y_data[first_kinase].shape)

X_data[first_kinase].head()

Kinase: O00418

X shape: (61, 5)
y shape: (61,)


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69)
Perturbation,,,,,
AC220,-0.669597,-1.389779,-0.107403,-0.118885,-0.230559
AT13148,-0.170124,1.309982,-2.477483,-2.136299,-1.968249
AZ20,-0.209610,-0.082364,-0.908485,-0.621383,-0.876680
AZD1480,-0.467337,0.795807,0.451792,0.352203,0.258493
AZD3759,-1.439665,-1.421963,0.442492,0.943424,-2.161574


In [25]:
# count missing values

missing_summary = {}

for kinase_id, X in X_data.items():
    
    missing_summary[kinase_id] = X.isna().sum().sum()

missing_df = (
    pd.DataFrame.from_dict(
        missing_summary,
        orient="index",
        columns=["Missing_FC_Values"] 
    )
    .reset_index()

)

missing_df.columns = ["UniprotID","Missing_FC_Values"]

missing_df.sort_values(
    by="Missing_FC_Values",
    ascending=False,
    inplace=True 
)

missing_df.head(10)

,UniprotID,Missing_FC_Values
0,O00418,0
54,Q15139,0
62,Q8IVT5,0
61,Q8IV63,0
60,Q7KZI7,0
59,Q2M2I8,0
58,Q16584,0
57,Q16513,0
56,Q16512,0
55,Q15418,0


In [26]:
missing_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 85 entries, 0 to 84
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   UniprotID          85 non-null     object
 1   Missing_FC_Values  85 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 2.0+ KB


In [27]:
print(missing_df.shape)

(85, 2)


In [28]:
#Feature scaling

X_scaled ={}

for kinase_id in X_data:

    scaler = StandardScaler()

    X_scaled[kinase_id] = pd.DataFrame(
        scaler.fit_transform(X_data[kinase_id]),
        columns=X_data[kinase_id].columns,
        index=X_data[kinase_id].index
    )

print(f"Scaled {len(X_scaled)} kinase feature matrices.")

Scaled 85 kinase feature matrices.


In [29]:
#inspect one kinase

first_kinase = list(X_scaled.keys())[0]

print(first_kinase)

X_scaled[first_kinase].head()

O00418


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69)
Perturbation,,,,,
AC220,-0.675162,-0.497620,0.240832,0.179318,0.277767
AT13148,0.089360,1.367409,-1.518386,-1.239949,-1.133377
AZ20,0.028920,0.405558,-0.353780,-0.174194,-0.246935
AZD1480,-0.365570,1.012210,0.655899,0.510732,0.674917
AZD3759,-1.853870,-0.519853,0.648997,0.926661,-1.290372


In [30]:
#Train/test split

X_train = {}
X_test = {}
y_train = {}
y_test = {}

for kinase in X_scaled:

    X_train[kinase], X_test[kinase], y_train[kinase], y_test[kinase] = train_test_split(
        X_scaled[kinase],
        y_data[kinase],
        test_size=0.3,
        random_state=42
    )

print(f"Prepared train/test sets for {len(X_train)} kinases.")

Prepared train/test sets for 85 kinases.


In [31]:
#verify one kinase

first_kinase = list(X_train.keys())[0]

print("Kinase:", first_kinase)

print("X_train:",X_train[first_kinase].shape)
print("X_test:", X_test[first_kinase].shape)

print("y_train:", y_train[first_kinase].shape)
print("y_test:", y_test[first_kinase].shape)

Kinase: O00418
X_train: (42, 5)
X_test: (19, 5)
y_train: (42,)
y_test: (19,)


In [32]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Cross-validation
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    # Train final model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return {
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_r2": r2,
        "test_mse": mse,
        "prediction": y_pred
    }

In [33]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

results = evaluate_model(
    rf,
    X_train[first_kinase],
    X_test[first_kinase],
    y_train[first_kinase],
    y_test[first_kinase]
)

print(results)

{'cv_mean': 0.22935212327825236, 'cv_std': 0.46778952472394475, 'test_r2': 0.44022514078173924, 'test_mse': 5.379335434310662, 'prediction': array([-0.9975738 , -0.09519968,  0.29785281,  0.09936514,  0.49634403,
        0.25416567, -0.10760778,  1.78483729,  2.36413785,  2.07782402,
        0.00695696,  0.03387029,  2.95969751, -0.19583879, -1.13503172,
        1.11851347,  2.90175519, -0.34326763,  2.20806796])}


In [34]:
def get_model(model_name, n_features=None):

    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )

    elif model_name == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

    elif model_name == "PLS":

        n_components = min(3, n_features)

        return PLSRegression(
            n_components=n_components
        )

    elif model_name == "SVR":

        return SVR(
            kernel="linear",
            C=1.0,
            epsilon=0.1
        )

    elif model_name == "Lasso":

        return Lasso(
            alpha=0.1
        )

    elif model_name == "ElasticNet":

        return ElasticNet(
            alpha=0.1,
            l1_ratio=0.5
        )

    elif model_name == "GradientBoosting":

        return GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

    elif model_name == "MLP":

        hidden_size = min(max(n_features - 1, 1), 10)

        return MLPRegressor(
            hidden_layer_sizes=(hidden_size,),
            activation="relu",
            solver="adam",
            max_iter=5000,
            random_state=1
        )

    else:

        raise ValueError(f"Unknown model: {model_name}")

In [35]:
#Test the function

first_kinase = list(X_train.keys())[0]

print(first_kinase)

n_features = X_train[first_kinase].shape[1]

print(n_features)

O00418
5


In [36]:
#Build one random forest model
rf = get_model(
    "RandomForest",
    n_features
)

print(rf)

RandomForestRegressor(random_state=42)


In [37]:
#Build one MLP model
mlp = get_model(
    "MLP",
    n_features
)

print(mlp)

MLPRegressor(hidden_layer_sizes=(4,), max_iter=5000, random_state=1)


In [38]:
#Build one XGBoost model
xgb = get_model(
    "XGBoost",
    n_features
)

print(xgb)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)


In [39]:
#Build one PLS model
pls = get_model(
    "PLS",
    n_features
)

print(pls)

PLSRegression(n_components=3)


In [40]:
#Build one SVR model 
svr = get_model(
    "SVR",
    n_features
)

print(svr)

SVR(kernel='linear')


In [41]:
#Build one Gradient Boosting model
gbr = get_model(
    "GradientBoosting",
    n_features
)

print(gbr)

GradientBoostingRegressor(random_state=42)


In [42]:
#Build one Lasso model 
lasso = get_model(
    "Lasso",
    n_features
)

print(lasso)

Lasso(alpha=0.1)


In [43]:
#Build one ElasticNet model
enet = get_model(
    "ElasticNet",
    n_features
)

print(enet)

ElasticNet(alpha=0.1)


In [44]:

#Evaluate one model on first kinase
# First kinase
first_kinase = list(X_train.keys())[0]

# Number of phosphosite features
n_features = X_train[first_kinase].shape[1]

# Choose model
model = get_model(
    "RandomForest",
    n_features
)

# Evaluate model
results = evaluate_model(
    model,
    X_train[first_kinase],
    X_test[first_kinase],
    y_train[first_kinase],
    y_test[first_kinase]
)

results



{'cv_mean': 0.22935212327825236,
 'cv_std': 0.46778952472394475,
 'test_r2': 0.44022514078173924,
 'test_mse': 5.379335434310662,
 'prediction': array([-0.9975738 , -0.09519968,  0.29785281,  0.09936514,  0.49634403,
         0.25416567, -0.10760778,  1.78483729,  2.36413785,  2.07782402,
         0.00695696,  0.03387029,  2.95969751, -0.19583879, -1.13503172,
         1.11851347,  2.90175519, -0.34326763,  2.20806796])}

In [45]:
results_df = pd.DataFrame({
    "Metric": [
        "CV Mean R²",
        "CV Std R²",
        "Test R²",
        "Test MSE"
    ],
    "Value": [
        results["cv_mean"],
        results["cv_std"],
        results["test_r2"],
        results["test_mse"]
    ]
})

results_df

,Metric,Value
0,CV Mean R²,0.229352
1,CV Std R²,0.467790
2,Test R²,0.440225
3,Test MSE,5.379335


In [46]:
#comparing all 8 models automatically on the first kinase
#create the list of models 
model_names = [
    "RandomForest",
    "XGBoost",
    "PLS",
    "SVR",
    "Lasso",
    "ElasticNet",
    "GradientBoosting",
    "MLP"
]

In [47]:
#Create an empty list

comparison_results =[]

In [48]:
#Loop through every model
for model_name in model_names:

    model = get_model(
        model_name,
        n_features
    )

    results = evaluate_model(
        model,
        X_train[first_kinase],
        X_test[first_kinase],
        y_train[first_kinase],
        y_test[first_kinase]
    )

    comparison_results.append({
        "Model": model_name,
        "CV Mean R²": results["cv_mean"],
        "CV Std R²": results["cv_std"],
        "Test R²": results["test_r2"],
        "Test MSE": results["test_mse"]
    })

In [49]:
#Convert to a DataFrame

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,RandomForest,0.229352,0.467790,0.440225,5.379335
1,XGBoost,-0.203245,0.697583,0.443658,5.346345
2,PLS,-0.876310,1.576216,0.171546,7.961298
3,SVR,-2.119992,3.662152,0.147104,8.196175
4,Lasso,-0.572028,1.013604,0.139454,8.269695
5,ElasticNet,-0.537889,1.001374,0.151534,8.153610
6,GradientBoosting,0.157575,0.567475,0.493956,4.862995
7,MLP,-0.491038,0.988228,0.343756,6.306388


In [50]:
#Sort from best to worst
comparison_df = comparison_df.sort_values(
    by="Test R²",
    ascending=False
)

comparison_df

,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
6,GradientBoosting,0.157575,0.567475,0.493956,4.862995
1,XGBoost,-0.203245,0.697583,0.443658,5.346345
0,RandomForest,0.229352,0.467790,0.440225,5.379335
7,MLP,-0.491038,0.988228,0.343756,6.306388
2,PLS,-0.876310,1.576216,0.171546,7.961298
5,ElasticNet,-0.537889,1.001374,0.151534,8.153610
3,SVR,-2.119992,3.662152,0.147104,8.196175
4,Lasso,-0.572028,1.013604,0.139454,8.269695


In [51]:
#Create an empty results Dataframe

all_results = pd.DataFrame(columns=[
    "Kinase",
    "Model",
    "CV Mean R²",
    "CV Std R²",
    "Test R²",
    "Test MSE"
])

In [52]:
#Create the list of model names

model_names = [
    "RandomForest",
    "XGBoost",
    "PLS",
    "SVR",
    "Lasso",
    "ElasticNet",
    "GradientBoosting",
    "MLP"
]

In [53]:
gene_lookup = (
    usable_df[["UniprotID", "GeneName"]]
    .drop_duplicates()
    .set_index("UniprotID")["GeneName"]
    .to_dict()
)

In [54]:
prediction_results = []

In [55]:
#Loop through all 85 kinases and all 8 models

for kinase in X_train.keys():

    print(f"\nProcessing kinase: {kinase}")

    # Number of phosphosite features for this kinase
    n_features = X_train[kinase].shape[1]

    # Loop through all models
    for model_name in model_names:

        print(f"   Running {model_name}...")

        # Create a fresh model
        model = get_model(model_name, n_features)

        # Evaluate the model
        results = evaluate_model(
            model,
            X_train[kinase],
            X_test[kinase],
            y_train[kinase],
            y_test[kinase]
        )

        for actual, predicted in zip(y_test[kinase],results["prediction"]):
            
            prediction_results.append({
                "Kinase": kinase,
                "GeneName": gene_lookup.get(kinase, "Unkown"),
                "Model": model_name,
                "Actual": actual,
                "Predicted": predicted 
            })

        # Save results
        all_results.loc[len(all_results)] = [
            kinase,
            model_name,
            results["cv_mean"],
            results["cv_std"],
            results["test_r2"],
            results["test_mse"]
        ]


Processing kinase: O00418
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O14578
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O15075
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O43318
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O43353
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running Gradient

In [56]:
prediction_df = pd.DataFrame(prediction_results)

prediction_df.to_csv(
    "all_predictions.csv",
    index=False
)

print(prediction_df.shape)

prediction_df.head()

(12904, 5)


,Kinase,GeneName,Model,Actual,Predicted
0,O00418,EEF2K,RandomForest,-6.199636,-0.997574
1,O00418,EEF2K,RandomForest,-4.673193,-0.095200
2,O00418,EEF2K,RandomForest,0.443448,0.297853
3,O00418,EEF2K,RandomForest,-0.263148,0.099365
4,O00418,EEF2K,RandomForest,-0.325892,0.496344


In [57]:
all_results.shape

(680, 6)

In [58]:
all_results

,Kinase,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,O00418,RandomForest,0.229352,0.467790,0.440225,5.379335
1,O00418,XGBoost,-0.203245,0.697583,0.443658,5.346345
2,O00418,PLS,-0.876310,1.576216,0.171546,7.961298
3,O00418,SVR,-2.119992,3.662152,0.147104,8.196175
4,O00418,Lasso,-0.572028,1.013604,0.139454,8.269695
...,...,...,...,...,...,...
675,Q9Y3S1,SVR,-0.155072,0.563966,-0.401132,3.445313
676,Q9Y3S1,Lasso,-0.051942,0.364465,-1.134598,5.248870
677,Q9Y3S1,ElasticNet,-0.020410,0.344429,-1.119894,5.212712
678,Q9Y3S1,GradientBoosting,-0.402917,1.274725,-4.595140,13.758167


In [59]:
all_results.to_csv(
    "all_models_85_kinases.csv",
    index=False 
)
print("Saved")

Saved


In [60]:
#which model wins for each kinase

best_models = (
    all_results
    .sort_values("Test R²", ascending=False)
    .groupby("Kinase")
    .first()
    .reset_index()
)

best_models

,Kinase,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,O00418,GradientBoosting,0.157575,0.567475,0.493956,4.862995
1,O14578,GradientBoosting,-0.427247,0.312659,0.412093,0.381244
2,O15075,RandomForest,-0.144578,0.457117,0.434111,1.128972
3,O43318,ElasticNet,-0.229869,0.216070,-0.056106,0.547607
4,O43353,PLS,0.364888,0.530895,0.851154,0.247871
...,...,...,...,...,...,...
80,Q9NYV4,SVR,-3.039157,3.882587,0.232942,0.372478
81,Q9UKE5,SVR,0.949891,0.021636,0.975446,0.010548
82,Q9Y2K2,Lasso,-0.173522,0.258623,0.052192,0.576382
83,Q9Y2U5,Lasso,-0.751310,0.646994,-0.012505,6.537153


In [61]:
#count how many kinases each model wins

best_models["Model"].value_counts()

Model
RandomForest        18
Lasso               14
SVR                 13
GradientBoosting    11
PLS                 10
ElasticNet           8
MLP                  8
XGBoost              3
Name: count, dtype: int64

In [62]:
best_models.to_csv(
    "best_model_per_kinase.csv",
    index=False
)